# Data agent: the indicator table

This notebook walks through what the data agent now produces: the **prices table** with **37 indicator
columns added on the right**. It reads the code, it does not change it.

| Step | File | What it does |
|---|---|---|
| 1. Prices | `data/data_agent.py` → `fetch_prices` | Daily OHLC, **adjusted close** and **volume** of the 50 ETFs, in EUR, on the Belfius exchange |
| 2. Quality | `data/data_agent.py` → `check_prices` | 8 reports (new: days without trade, adjusted closes) |
| 3. Indicators | `data/features.py` → `add_indicators` | Same table + 37 raw indicators, ETF by ETF |
| 4. Shared format | `data/features.py` → `compute_features` | Same values in the long Features format, for the pipeline |

**Two floors.** This notebook covers floor 1: the data agent *calculates* raw indicators (moving averages and highs/lows
stay in euros). **All indicators use the adjusted close** (`adj_close_eur`): on an ex-dividend day the raw close of a
distributing ETF drops, but that is not a market move. So indicator levels are in *adjusted* euros: compare them
with `adj_close_eur`, never with `close_eur`. `close_eur` stays the price for valuation and orders. Floor 2, the technical agent (`agents/technical.py`), will *read* them: price / SMA 200 − 1, distance
to a high, trend of the OBV, rules per family, score from −1 to +1. Section 6 shows a preview of that reading.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import inspect
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

sys.path.insert(0, os.path.abspath(".."))   # repo root, to import data/
from data import data_agent, features
from data.config import DATA_DIR

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 1. The prices

`SOURCE` decides where the prices come from:
- `"fetch"`: download them now (Yahoo + EODHD, about a minute). **The only option with the volume and the adjusted close.** The table is saved in `data/cache/`.
- `"cache"`: the latest `data/cache/prices_ohlc_*.csv`. Fast, but files written before 12/11 have no volume and no adjusted close (indicators then fall back to the raw close) and may predate the split fixes.
- `"mock"`: random prices from `data/mock.py`, without network.

In [ ]:
SOURCE = "fetch"      # "fetch", "cache" or "mock"

if SOURCE == "fetch":
    end = date.today() - timedelta(days=1)
    prices = data_agent.fetch_prices(date(2020, 1, 1), end)
    cache_file = DATA_DIR / "cache" / f"prices_ohlc_{end.isoformat()}.csv"
    cache_file.parent.mkdir(exist_ok=True)
    prices.to_csv(cache_file, index=False)
    print(f"Saved to {cache_file}")
elif SOURCE == "cache":
    files = sorted((DATA_DIR / "cache").glob("prices_ohlc_*.csv"))
    print(f"Reading {files[-1].name}")
    prices = pd.read_csv(files[-1])
    if "filled" in prices.columns:
        prices = prices[prices["filled"] != True]   # old cache format: drop the filled rows, keep real quotes only
    if "adj_close_eur" not in prices.columns:
        prices["adj_close_eur"] = np.nan   # old cache: indicators fall back to the raw close
    if "volume" not in prices.columns:
        prices["volume"] = np.nan          # old cache without volume: nothing is invented
    prices = prices[data_agent.PRICES]
else:
    from data.mock import mock_prices
    prices = mock_prices(days=600)

prices["date"] = pd.to_datetime(prices["date"]).dt.date
print(f"{prices['ticker'].nunique()} ETFs, {len(prices)} rows, from {prices['date'].min()} to {prices['date'].max()}")
prices.tail()

### 1.1 The new quality reports: days without trade, adjusted closes

Volume = units traded **on the exchange only** (an ETF also trades off-exchange), so it is partial.
A volume of 0 means no trade that day: the "close" is probably the previous one repeated.
This is a data-quality check, not an indicator.

The `adjusted` report lists the ETFs whose adjusted close differs from the close: expected for the distributing
ETFs (dividends), to look at for an accumulating one. `oldest_ratio` = adjusted / real close on the first day.

In [ ]:
reports = data_agent.check_prices(prices) if SOURCE != "mock" else {}
if "no_volume" in reports:
    no_volume = reports["no_volume"].sort_values("zero_volume_percent", ascending=False)
    print(f"{len(no_volume)} ETFs with at least one day without trade or without volume data (last year)")
    display(no_volume.head(15))
if "adjusted" in reports:
    print("ETFs whose adjusted close differs from the close (dividends) or is missing:")
    display(reports["adjusted"])

## 2. Add the indicators

One call. Inside, `add_indicators` loops over the ETFs: for each one it takes its own rows in date order,
calls one small function per indicator, and glues the results as new columns.

In [ ]:
table = features.add_indicators(prices)
print(f"{table.shape[0]} rows x {table.shape[1]} columns "
      f"({len(data_agent.PRICES)} price columns + {len(features.FEATURE_COLUMNS)} indicators)")

for block, columns in features.FEATURE_BLOCKS.items():
    print(f"{block:20} {', '.join(columns)}")

## 3. One ETF, last day: the 37 values with how to read them

Change `TICKER` to look at another ETF (Belfius short names: `WORLD`, `GOLD`, `SEMIS`, `BEL 20`...).

In [ ]:
TICKER = "WORLD"

HOW_TO_READ = {
    "sma_5": "mean of the last 5 closes (EUR)", "sma_10": "mean of the last 10 closes (EUR)",
    "sma_20": "short-term trend (EUR)", "sma_50": "medium-term trend (EUR)", "sma_200": "long-term trend (EUR)",
    "ema_5": "like SMA 5, recent days weigh more", "ema_10": "like SMA 10, more reactive",
    "ema_20": "like SMA 20, more reactive", "sma_50_slope_10": "SMA 50 vs 10 days ago (> 0: rising)",
    "ret_10d": "return over 10 quotes", "ret_1m": "return over 21 quotes", "ret_3m": "return over 63 quotes",
    "ret_6m": "return over 126 quotes", "ret_12m_ex_1m": "return over 1 year, last month skipped",
    "rsi_2": "< 10: very short-term oversold", "rsi_5": "< 20: oversold over a week",
    "rsi_14": "> 70 overbought, < 30 oversold", "zscore_20": "std devs from the 20-day mean",
    "efficiency_ratio_20": "1 = straight line, 0 = back and forth", "adx_14": "> 25: clear trend, < 20: none",
    "high_20": "highest close of 20 days (EUR)", "low_20": "lowest close of 20 days (EUR)",
    "high_52w": "highest close of 1 year (EUR)", "low_52w": "lowest close of 1 year (EUR)",
    "vol_20": "annualised volatility, 1 month", "vol_60": "annualised volatility, 3 months",
    "vol_ratio_20_60": "> 1: the ETF is getting more agitated", "downside_vol_60": "volatility of down days only",
    "atr_14": "usual size of a day's move (EUR)", "bb_upper_20": "upper Bollinger band (EUR)",
    "bb_lower_20": "lower Bollinger band (EUR)", "drawdown_60": "loss from the 60-day high",
    "sharpe_63": "steadiness of the 3-month move", "worst_day_60": "worst daily return, 60 days",
    "volume_rel_20": "today's volume / 20-day mean", "obv": "running volume (read its trend)",
    "volume_trend_20_60": "> 1: more trading than usual",
}

etf = table[table["ticker"] == TICKER].reset_index(drop=True)
last = etf.iloc[-1]
print(f"{TICKER} on {last['date']}: close {last['close_eur']:.2f} EUR, adjusted close {last['adj_close_eur']:.2f} EUR")

rows = []
for block, columns in features.FEATURE_BLOCKS.items():
    for column in columns:
        rows.append({"block": block, "indicator": column, "value": last[column], "how to read": HOW_TO_READ[column]})
pd.DataFrame(rows).style.format({"value": "{:.4f}"}).hide(axis="index")

## 4. Block by block, on a chart

Last 300 quotes of `TICKER`. Each chart shows one block of indicators next to the price.

In [ ]:
recent = etf.tail(300).copy()
recent["date"] = pd.to_datetime(recent["date"])


def line_chart(df, columns, title, height=220):
    """One line per column, same axis."""
    long = df[["date"] + columns].melt("date", var_name="series", value_name="value").dropna()
    return alt.Chart(long, title=title).mark_line().encode(
        x="date:T", y=alt.Y("value:Q", scale=alt.Scale(zero=False)), color="series:N"
    ).properties(width=800, height=height)


line_chart(recent, ["adj_close_eur", "sma_20", "sma_50", "sma_200", "bb_upper_20", "bb_lower_20"],
           f"{TICKER}: adjusted close, moving averages and Bollinger bands")

In [ ]:
line_chart(recent, ["adj_close_eur", "high_20", "low_20", "high_52w", "low_52w"],
           f"{TICKER}: support and resistance (a close on high_20 = a breakout)")

In [ ]:
alt.vconcat(
    line_chart(recent, ["ret_10d", "ret_1m", "ret_3m", "ret_6m", "ret_12m_ex_1m"], "Momentum: returns on 5 horizons", 180),
    line_chart(recent, ["rsi_2", "rsi_14"], "Oscillators: RSI 2 and RSI 14 (70 / 30 = overbought / oversold)", 180),
    line_chart(recent, ["zscore_20"], "Z-score 20 (beyond +/-2 = outside the Bollinger bands)", 140),
    line_chart(recent, ["efficiency_ratio_20"], "Trend strength: efficiency ratio (0 to 1)", 140),
    line_chart(recent, ["adx_14"], "Trend strength: ADX (> 25 = clear trend)", 140),
)

In [ ]:
alt.vconcat(
    line_chart(recent, ["vol_20", "vol_60", "downside_vol_60"], "Volatility (annualised)", 180),
    line_chart(recent, ["vol_ratio_20_60"], "Volatility ratio 20 / 60 (> 1 = waking up)", 140),
    line_chart(recent, ["drawdown_60", "worst_day_60"], "Risk: drawdown from the 60-day high, worst day", 180),
    line_chart(recent, ["sharpe_63"], "Risk: steadiness of the move (Sharpe 63 days)", 140),
)

In [ ]:
if recent["volume"].notna().any():
    display(alt.vconcat(
        alt.Chart(recent, title="Volume (units traded on the exchange)").mark_bar().encode(x="date:T", y="volume:Q")
        .properties(width=800, height=140),
        line_chart(recent, ["volume_rel_20", "volume_trend_20_60"], "Relative volume and volume trend", 160),
        line_chart(recent, ["obv"], "OBV: its level means nothing, its trend does", 160),
    ))
else:
    print("No volume in this price table (SOURCE = 'cache' with an old file?): volume indicators are empty.")

## 5. Inside one indicator: the RSI

Every indicator is one small function in `data/features.py`. Here is the RSI, then the same calculation
done by hand on 4 closes, to check it.

In [ ]:
print(inspect.getsource(features.compute_rsi))
print(inspect.getsource(features.wilder_average))

In [ ]:
closes = pd.Series([1.0, 2.0, 3.0, 2.0])
# changes: +1, +1, -1
# Wilder average over 2 days (keep half of yesterday, add half of today):
#   gains:  1, then 0.5 x 1 + 0.5 x 1 = 1, then 0.5 x 1 + 0.5 x 0 = 0.5
#   losses: 0, then 0,                     then 0.5 x 0 + 0.5 x 1 = 0.5
# day 3: no loss -> RSI 100. Day 4: gains = losses -> RSI 50.
print(features.compute_rsi(closes, 2).tolist())

## 6. Preview of floor 2: reading the table on the last day

Not part of the data agent. Example of what the technical agent will do: turn raw levels into comparable numbers
(price / SMA 200 − 1, distance to the 1-year high) and compare the ETFs. Sorted by 3-month return.

In [ ]:
last_day = table.groupby("ticker").tail(1).copy()
# Indicator levels are in adjusted euros: compare them with the adjusted close
last_day["price_vs_sma200"] = last_day["adj_close_eur"] / last_day["sma_200"] - 1
last_day["dist_52w_high"] = last_day["adj_close_eur"] / last_day["high_52w"] - 1
last_day["new_20d_high"] = last_day["adj_close_eur"] >= last_day["high_20"]

columns = ["ticker", "date", "ret_3m", "ret_12m_ex_1m", "price_vs_sma200", "dist_52w_high", "new_20d_high",
           "rsi_14", "adx_14", "vol_20", "drawdown_60"]
(last_day[columns].sort_values("ret_3m", ascending=False)
 .style.format({c: "{:.1%}" for c in ["ret_3m", "ret_12m_ex_1m", "price_vs_sma200", "dist_52w_high", "vol_20", "drawdown_60"]}
              | {"rsi_14": "{:.0f}", "adx_14": "{:.0f}"})
 .hide(axis="index"))

## 7. The shared format for the pipeline

`compute_features` gives the same values in long format (`date, ticker, feature, value`): one row per value,
empty values dropped. It is what `data_agent.compute_features` returns to `graph/pipeline.py`.

In [ ]:
long = data_agent.compute_features(prices)
print(f"{len(long)} rows, {long['feature'].nunique()} features")
display(long[(long["ticker"] == TICKER)].tail(8))

# Back to the wide table, if an agent prefers it
wide_again = long.pivot_table(index=["date", "ticker"], columns="feature", values="value")
wide_again.tail(3)

## 8. What comes next

1. **Floor 2**: `agents/strategy_config.py` (blocks, weights, rules per family) and `technical_det` in `agents/technical.py`.
2. **Ranking table** of the 50 ETFs for the mid-term report (22/11): section 6 is a first draft.
3. **Backtest**: which indicators really help, family by family.
4. Set `USE_MOCK["data"] = False` in `data/config.py` once the real run looks right.